# Receipt extraction: train on A100, infer locally

Run this notebook on **one A100 (40 GB or 80 GB)** with `dataset/SROIE2019` available relative to the working directory. Use a CUDA-enabled PyTorch installation; install `transformers peft accelerate pillow` in the notebook's Python environment.

Training uses BF16 LoRA, batch size 4, and two accumulation steps (effective batch size 8). The base model and 768-pixel image setting stay the same as the 4 GB workflow so deployment keeps its existing memory budget. A larger training GPU does not require a larger inference model. Gradient checkpointing is optional; enable it or lower `BATCH_SIZE` if needed. These are starting settings, not measured A100 memory guarantees.

For a quick end-to-end check, set `MAX_STEPS = 5`. After training, copy the entire exported `smolvlm-receipt-lora/` directory to your local device and use `inference_local.py`. The local loader quantizes the base to NF4 and attaches the saved adapter; no merged BF16 model is needed. Quantized inference can change accuracy, so validate the exported adapter on the local GPU as well.

In [ ]:
import json
import math
import random
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import DataLoader
from transformers import AutoProcessor, AutoModelForImageTextToText, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model

MODEL_ID = "HuggingFaceTB/SmolVLM-Base"
DATA = Path("dataset/SROIE2019")
OUTPUT_DIR = Path("smolvlm-receipt-lora")
SEED = 42
MAX_STEPS = None       # optimizer steps; set to 5 for a smoke test
IMG_LONGEST_EDGE = 768 # preserve the local 4 GB inference preprocessing
BATCH_SIZE = 4         # starting point for a single A100 40 GB; also usable on 80 GB
GRAD_ACCUM = 2         # effective batch size = 4 * 2 = 8 receipts
EPOCHS, LR = 2, 1e-4
GRADIENT_CHECKPOINTING = False  # enable to trade speed for lower training memory
TRAIN_DTYPE = torch.bfloat16

if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("Run training on a CUDA GPU with BF16 support (intended: A100). Use inference_local.py locally.")
if BATCH_SIZE < 1 or GRAD_ACCUM < 1 or EPOCHS < 1:
    raise ValueError("BATCH_SIZE, GRAD_ACCUM and EPOCHS must be positive integers.")
if MAX_STEPS is not None and MAX_STEPS < 1:
    raise ValueError("MAX_STEPS must be None or a positive integer.")
DEVICE = torch.device("cuda:0")
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True

props = torch.cuda.get_device_properties(DEVICE)
print(f"Training GPU: {props.name}, {props.total_memory / 2**30:.1f} GiB")

processor = AutoProcessor.from_pretrained(MODEL_ID, do_image_splitting=True)
processor.image_processor.size = {"longest_edge": IMG_LONGEST_EDGE}
processor.tokenizer.padding_side = "right"

# A100 has room for the BF16 base (~4.4 GB of weights); quantize only at local inference.
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=TRAIN_DTYPE,
    attn_implementation="sdpa",
    device_map={"": 0},
)
model.config.use_cache = False
if hasattr(model.config, "text_config"):
    model.config.text_config.use_cache = False
print(f"GPU memory after load: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

In [ ]:
QUESTION = "Extract company, date, address and total from this receipt as JSON."

def load_split(split):
    rows = []
    for ent in sorted((DATA / split / "entities").glob("*.txt")):
        img = DATA / split / "img" / (ent.stem + ".jpg")
        if img.exists():
            rows.append({"image": img, "answer": json.dumps(json.loads(ent.read_text()), ensure_ascii=False)})
    if not rows:
        raise FileNotFoundError(f"No receipt/entity pairs found in {DATA / split}. Upload or unpack the dataset on the node.")
    return rows

train_rows, test_rows = load_split("train"), load_split("test")
print(len(train_rows), "train /", len(test_rows), "test")
print(train_rows[0]["answer"])

def build_prompt():
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": QUESTION}]}]
    return processor.apply_chat_template(msgs, add_generation_prompt=True)

print(repr(build_prompt()))

In [ ]:
# Standard LoRA: k-bit preparation is only needed for quantized training.
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    # Only the language model's linear layers; the vision encoder stays frozen.
    target_modules=r".*text_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.print_trainable_parameters()

image_token_id = processor.tokenizer.convert_tokens_to_ids("<image>")

def collate(rows):
    """Build CPU batches; loss only on answer tokens, excluding prompt/image/padding."""
    images = []
    for row in rows:
        with Image.open(row["image"]) as image:
            images.append([image.convert("RGB")])
    prompt = build_prompt()
    fulls = [prompt + r["answer"] + "<end_of_utterance>\n" for r in rows]
    batch = processor(text=fulls, images=images, padding=True, return_tensors="pt")
    prompt_batch = processor(text=[prompt] * len(rows), images=images, padding=True, return_tensors="pt")
    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100
    labels[labels == image_token_id] = -100
    for i in range(len(rows)):
        labels[i, : int(prompt_batch["attention_mask"][i].sum())] = -100
    batch["labels"] = labels
    return batch

def to_device(batch):
    return {
        k: v.to(device=DEVICE, dtype=TRAIN_DTYPE if v.is_floating_point() else v.dtype, non_blocking=True)
        for k, v in batch.items()
    }

b = collate(train_rows[:BATCH_SIZE])
print({k: tuple(v.shape) for k, v in b.items()}, "| trained tokens:", int((b["labels"] != -100).sum()))
print(processor.tokenizer.decode(b["labels"][0][b["labels"][0] != -100]))
del b

In [ ]:
train_loader = DataLoader(
    train_rows, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
    pin_memory=True, num_workers=0,  # safe in notebooks; returns CPU tensors
    generator=torch.Generator().manual_seed(SEED),
)
params = [p for p in model.parameters() if p.requires_grad]
optim = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = EPOCHS * steps_per_epoch
if MAX_STEPS is not None:
    total_steps = min(MAX_STEPS, total_steps)
warmup_steps = min(5, max(0, total_steps - 1))
sched = get_linear_schedule_with_warmup(optim, warmup_steps, total_steps)
print(f"{total_steps} optimizer steps; effective batch size {BATCH_SIZE * GRAD_ACCUM}")

model.train()
model.config.use_cache = False
optim.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()
step, running, logged_steps = 0, 0.0, 0
for epoch in range(EPOCHS):
    group_loss = 0.0
    for batch_idx, cpu_batch in enumerate(train_loader):
        # Weight by receipts so the last, smaller accumulation group is not under-scaled.
        group_start = (batch_idx // GRAD_ACCUM) * GRAD_ACCUM * BATCH_SIZE
        group_examples = min(BATCH_SIZE * GRAD_ACCUM, len(train_rows) - group_start)
        batch = to_device(cpu_batch)
        weight = batch["input_ids"].shape[0] / group_examples
        with torch.autocast(device_type="cuda", dtype=TRAIN_DTYPE):
            raw_loss = model(**batch).loss
            loss = raw_loss * weight
        loss.backward()
        group_loss += raw_loss.detach().float().item() * weight
        del batch, cpu_batch, loss, raw_loss

        end_group = (batch_idx + 1) % GRAD_ACCUM == 0 or batch_idx + 1 == len(train_loader)
        if end_group:
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optim.step()
            sched.step()
            optim.zero_grad(set_to_none=True)
            step += 1
            running += group_loss
            logged_steps += 1
            group_loss = 0.0
            if step % 5 == 0 or step == 1 or step == total_steps:
                print(f"epoch {epoch + 1} step {step}/{total_steps} loss {running / logged_steps:.4f} "
                      f"peak memory {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
                running, logged_steps = 0.0, 0
            if step >= total_steps:
                break
    if step >= total_steps:
        break

# Release optimizer state before evaluation. BF16 does not require an FP16 GradScaler.
del optim, sched
torch.cuda.empty_cache()

In [ ]:
# Export before evaluation so the trained adapter is available even if evaluation is interrupted.
import importlib.metadata

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
deployment = {
    "model_id": MODEL_ID,
    "question": QUESTION,
    "image_longest_edge": IMG_LONGEST_EDGE,
    "max_new_tokens": 200,
    "training_precision": "bfloat16",
    "local_quantization": "nf4",
}
(OUTPUT_DIR / "deployment_config.json").write_text(json.dumps(deployment, indent=2) + "\n")
versions = {name: importlib.metadata.version(name) for name in ("torch", "transformers", "peft", "accelerate", "Pillow")}
(OUTPUT_DIR / "package_versions.json").write_text(json.dumps(versions, indent=2) + "\n")
print(f"Copy all of {OUTPUT_DIR}/ to the local device, together with inference_local.py.")

In [ ]:
@torch.inference_mode()
def predict(row):
    model.eval()
    with Image.open(row["image"]) as image:
        img = image.convert("RGB")
    inputs = to_device(processor(text=[build_prompt()], images=[[img]], return_tensors="pt"))
    with torch.autocast(device_type="cuda", dtype=TRAIN_DTYPE):
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False, use_cache=True)
    text = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    try:
        result = json.loads(text.strip())
        return result if isinstance(result, dict) else {}
    except json.JSONDecodeError:
        return {}

N_EVAL = 50
fields = ["company", "date", "address", "total"]
hits = {f: 0 for f in fields}
sample = test_rows[:N_EVAL]
for row in sample:
    pred, gold = predict(row), json.loads(row["answer"])
    for f in fields:
        hits[f] += str(pred.get(f, "")).strip().lower() == str(gold.get(f, "")).strip().lower()
print("A100 BF16 exact-match results (repeat evaluation locally with 4-bit inference):")
print({f: f"{hits[f]/len(sample):.0%}" for f in fields})

## Run on your local 4 GB GPU

Install CUDA-enabled PyTorch and `transformers peft accelerate bitsandbytes pillow`. Use the exported `package_versions.json` to match the training environment where possible. The base model downloads from Hugging Face on first use (or uses your existing cache).

```bash
python inference_local.py path/to/receipt.jpg --adapter smolvlm-receipt-lora
```

The script processes one receipt at a time at the saved 768-pixel setting, loads the base in 4-bit NF4, and selects BF16 or FP16 compute according to local GPU support. It prints the generated receipt JSON. Runtime memory still depends on the receipt and generated sequence; if needed, try `--max-new-tokens 128` first, or `--image-longest-edge 384` (which may reduce OCR accuracy). The A100 training model stays on the node; inference runs on the local GPU.

API references: [SmolVLM](https://huggingface.co/docs/transformers/model_doc/smolvlm), [PEFT quantization](https://huggingface.co/docs/peft/en/developer_guides/quantization).